# Week 4: Final Rigorous Validation of Stage-2 Selective Evidence Allocation (Policy C1)

**Project:** Query-Conditioned Context Allocator (QCCA) for Efficient Retrieval-Augmented Generation  
**Parent Architecture:** SARA — Selective and Adaptive Retrieval-augmented Generation with Context Compression  
**Objective:** Rigorous final empirical validation of Policy C1:
> *"Can pre-generation retrieval/semantic/coverage features reliably identify when expanding evidence from $k=4$ to $k=6$ is useful, enabling near-static-$k=6/k=8$ answer quality at lower context cost?"*

### Strict Governance Controls:
1. **Zero Test Set Access:** Development split only (86 papers, 231 queries). Historical test split remains 100% frozen and untouched.
2. **Strict Paper-Disjoint GroupKFold:** All modeling, scaling, and threshold evaluations occur strictly out-of-fold grouped by `paper_id`.
3. **No Feature Leakage:** Only deployable pre-generation observables are used.
4. **No Model Shopping:** Evaluates a single fixed primary model (Balanced Logistic Regression) against standard baselines.


In [1]:
import os
import sys
from pathlib import Path

# Robust project root discovery (supports running from repo root or notebooks/week4)
_curr = Path.cwd().resolve()
candidates = [_curr, _curr.parent, _curr.parent.parent]
PROJECT_ROOT = next((p for p in candidates if (p / "src").is_dir() and (p / "results").is_dir()), _curr)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
print(f"Working directory set to Project Root: {PROJECT_ROOT}")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.week4.qcca_v2 import (
    CONTEXT_TOKEN_COSTS,
    PROVISIONAL_14_FEATURES,
    load_qcca_v2_data,
    compute_pareto_frontier,
)
from src.week4.qcca_v2_final_validation import (
    STAGE_2_CORE_FEATURES,
    STAGE_2_FEATURE_JUSTIFICATIONS,
    simulate_policy_c1,
    run_stage2_oof_evaluation,
    evaluate_stage2_threshold_grid,
    run_stage2_controlled_ablations,
    evaluate_stage2_subgroups,
    perform_stage2_error_analysis,
    run_stage2_statistical_comparisons,
    generate_stage2_validation_figures,
)

OUTPUT_DIR = PROJECT_ROOT / "results/week4/qcca_v2_final_validation"
FIG_DIR = OUTPUT_DIR / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("Environment initialized successfully. Output paths configured.")


Working directory set to Project Root: /home/gunavenkat/Downloads/CS787-RAG-Project
Environment initialized successfully. Output paths configured.


## 1. Data Loading & Feature Set Specification
Loads the 231 QASPER development queries across 86 papers with frozen response surfaces ($k \in \{2, 4, 6, 8\}$) and deployable pre-generation features.


In [2]:
df_f, df_t, df_merged = load_qcca_v2_data(
    feature_matrix_path="results/week4/feature_discovery/feature_matrix.csv",
    dev_matrix_path="results/week2/processed/dev_per_query_k_matrix.csv",
    oracle_path="results/week3/processed/epsilon_oracle_all.csv",
)
print(f"Loaded {len(df_merged)} queries across {df_merged['paper_id'].nunique()} papers.")
print(f"Stage 2 Target (Y_4_6) Positive Rate: {df_merged['Y_4_6'].mean()*100:.1f}% ({df_merged['Y_4_6'].sum()}/{len(df_merged)})")

print("\nValidated Stage-2 Core Features and Scientific Justifications:")
for feat, just in STAGE_2_FEATURE_JUSTIFICATIONS.items():
    print(f"- {feat}: {just}\n")


Loaded 231 queries across 86 papers.
Stage 2 Target (Y_4_6) Positive Rate: 19.5% (45/231)

Validated Stage-2 Core Features and Scientific Justifications:
- sem_sim_mean_top6: Measures semantic coherence across the candidate retrieval window tail. High semantic similarity in ranks 4-6 indicates that additional passages contain on-topic scientific evidence rather than unrelated background noise.

- bm25_mean_score: Measures background keyword matching density. Higher average BM25 signals that the document contains dense relevant scientific terminology, justifying broader passage retrieval.

- lex_coverage_top2: Cumulative lexical query term coverage across top-2 passages. Moderately high coverage combined with remaining uncovered terms indicates that expanding to k=6 will capture missing factual facets.

- evidence_query_cluster_span: Number of distinct document sections/clusters spanned by query keywords. Multi-section evidence dispersion requires k >= 6 to capture physically disconnect

## 2. Rebuild Strict Out-of-Fold Stage-2 Evaluation
Produces genuine out-of-fold predictions for every query under 5-fold paper-disjoint GroupKFold cross-validation. No query receives a prediction from a model trained on its paper.


In [3]:
res_primary = run_stage2_oof_evaluation(
    df_merged,
    feature_cols=STAGE_2_CORE_FEATURES,
    target_col="Y_4_6",
    gain_col="G_4_6",
    model_type="logistic_regression_balanced",
    n_splits=5,
    seed=42,
)

df_stage2_oof = res_primary["df_predictions"]
df_stage2_oof.to_csv(OUTPUT_DIR / "stage2_oof_predictions.csv", index=False)

print("Stage 2 OOF Predictions successfully saved to results/week4/qcca_v2_final_validation/stage2_oof_predictions.csv")
display(df_stage2_oof.head(10))


Stage 2 OOF Predictions successfully saved to results/week4/qcca_v2_final_validation/stage2_oof_predictions.csv


,question_id,paper_id,p_expand_4_6,pred_class_4_6,selected_k,f1_k4,f1_k6,actual_gain_4_6,y_true_4_6,policy_f1,context_tokens,regret,fold_id
0,1912,638,0.2622,0,4,0.0000,0.0000,0.0000,0,0.0000,946.1,0.5000,2
1,2201,798,0.6856,1,6,0.6316,0.0000,-0.6316,0,0.0000,1359.1,0.8000,2
2,883,254,0.3997,0,4,0.6667,1.0000,0.3333,1,0.6667,946.1,0.3333,2
3,884,254,0.2138,0,4,1.0000,0.8889,-0.1111,0,1.0000,946.1,0.0000,2
4,104,28,0.3074,0,4,0.0000,0.0000,0.0000,0,0.0000,946.1,0.0000,4
5,105,28,0.4766,0,4,0.4444,0.4444,0.0000,0,0.4444,946.1,0.0000,4
6,2100,735,0.4736,0,4,1.0000,0.0000,-1.0000,0,1.0000,946.1,0.0000,2
7,2101,735,0.4916,0,4,1.0000,1.0000,0.0000,0,1.0000,946.1,0.0000,2
8,2102,735,0.3004,0,4,0.0000,0.0000,0.0000,0,0.0000,946.1,0.0000,2
9,732,203,0.3736,0,4,1.0000,1.0000,0.0000,0,1.0000,946.1,0.0000,2


## 3. Fixed Primary Model vs Baselines (No Model Shopping)
Evaluates the primary model (Balanced Logistic Regression) against standard baselines (Majority Prior and Unweighted Logistic Regression).


In [4]:
res_majority = run_stage2_oof_evaluation(
    df_merged, STAGE_2_CORE_FEATURES, "Y_4_6", "G_4_6", model_type="majority", n_splits=5, seed=42
)
res_unweighted = run_stage2_oof_evaluation(
    df_merged, STAGE_2_CORE_FEATURES, "Y_4_6", "G_4_6", model_type="logistic_regression", n_splits=5, seed=42
)

models_comparison = [
    {
        "model_name": "Majority Baseline (Always Stop k=4)",
        "roc_auc": res_majority["roc_auc"],
        "pr_auc": res_majority["pr_auc"],
        "balanced_acc": res_majority["balanced_acc"],
        "precision": res_majority["precision"],
        "recall": res_majority["recall"],
        "f1": res_majority["f1"],
        "brier_score": res_majority["brier_score"],
        "ece": res_majority["ece"],
        "policy_mean_f1": res_majority["policy_summary"]["mean_f1"],
        "mean_k": res_majority["policy_summary"]["mean_k"],
        "mean_tokens": res_majority["policy_summary"]["mean_context_tokens"],
        "token_reduction_vs_k8": res_majority["policy_summary"]["context_reduction_pct_vs_k8"],
    },
    {
        "model_name": "Logistic Regression (Unweighted, t=0.5)",
        "roc_auc": res_unweighted["roc_auc"],
        "pr_auc": res_unweighted["pr_auc"],
        "balanced_acc": res_unweighted["balanced_acc"],
        "precision": res_unweighted["precision"],
        "recall": res_unweighted["recall"],
        "f1": res_unweighted["f1"],
        "brier_score": res_unweighted["brier_score"],
        "ece": res_unweighted["ece"],
        "policy_mean_f1": res_unweighted["policy_summary"]["mean_f1"],
        "mean_k": res_unweighted["policy_summary"]["mean_k"],
        "mean_tokens": res_unweighted["policy_summary"]["mean_context_tokens"],
        "token_reduction_vs_k8": res_unweighted["policy_summary"]["context_reduction_pct_vs_k8"],
    },
    {
        "model_name": "Logistic Regression (Balanced, Primary)",
        "roc_auc": res_primary["roc_auc"],
        "pr_auc": res_primary["pr_auc"],
        "balanced_acc": res_primary["balanced_acc"],
        "precision": res_primary["precision"],
        "recall": res_primary["recall"],
        "f1": res_primary["f1"],
        "brier_score": res_primary["brier_score"],
        "ece": res_primary["ece"],
        "policy_mean_f1": res_primary["policy_summary"]["mean_f1"],
        "mean_k": res_primary["policy_summary"]["mean_k"],
        "mean_tokens": res_primary["policy_summary"]["mean_context_tokens"],
        "token_reduction_vs_k8": res_primary["policy_summary"]["context_reduction_pct_vs_k8"],
    },
]

df_calib_models = pd.DataFrame(models_comparison)
df_calib_models.to_csv(OUTPUT_DIR / "calibration_results.csv", index=False)
display(df_calib_models)


,model_name,roc_auc,pr_auc,balanced_acc,precision,recall,f1,brier_score,ece,policy_mean_f1,mean_k,mean_tokens,token_reduction_vs_k8
0,Majority Baseline (Always Stop k=4),0.5000,0.1948,0.5000,0.0000,0.0000,0.0000,0.1948,0.1948,0.3596,4.00,946.1,46.20
1,"Logistic Regression (Unweighted, t=0.5)",0.6532,0.3056,0.4946,0.0000,0.0000,0.0000,0.1509,0.0206,0.3596,4.02,949.7,46.00
2,"Logistic Regression (Balanced, Primary)",0.6503,0.3199,0.6448,0.3053,0.6444,0.4143,0.2307,0.2687,0.3842,4.82,1115.9,36.54


## 4. Decision Threshold Analysis ($t \in [0.20, 0.80]$)
Evaluates Policy C1 across decision thresholds on downstream answer F1, context tokens, regret, and differences relative to Static $k \in \{4, 6, 8\}$.


In [5]:
df_threshold_sens = evaluate_stage2_threshold_grid(
    df_merged,
    res_primary["oof_probs"],
    thresholds=np.linspace(0.20, 0.80, 13),
)
df_threshold_sens.to_csv(OUTPUT_DIR / "threshold_sensitivity.csv", index=False)
display(df_threshold_sens)


,threshold,fraction_expanded_k6,mean_f1,mean_k,mean_context_tokens,context_reduction_pct_vs_k8,mean_regret,median_regret,fraction_zero_regret,delta_f1_vs_k4,delta_f1_vs_k6,delta_f1_vs_k8
0,0.20,91.8,0.3829,5.84,1325.1,24.64,0.1066,0.0,66.67,0.0233,-0.0028,-0.0121
1,0.25,88.3,0.3862,5.77,1310.8,25.46,0.1033,0.0,67.53,0.0266,0.0005,-0.0088
2,0.30,82.3,0.3815,5.65,1285.8,26.88,0.1080,0.0,67.53,0.0219,-0.0042,-0.0135
3,0.35,74.9,0.3792,5.50,1255.4,28.61,0.1103,0.0,68.40,0.0196,-0.0065,-0.0158
4,0.40,64.9,0.3773,5.30,1214.3,30.95,0.1122,0.0,68.40,0.0177,-0.0084,-0.0177
5,0.45,55.8,0.3752,5.12,1176.7,33.08,0.1143,0.0,67.53,0.0156,-0.0105,-0.0198
6,0.50,41.1,0.3842,4.82,1115.9,36.54,0.1053,0.0,69.70,0.0246,-0.0015,-0.0108
7,0.55,29.9,0.3743,4.60,1069.5,39.18,0.1152,0.0,68.40,0.0147,-0.0114,-0.0207
8,0.60,20.8,0.3726,4.42,1031.9,41.32,0.1169,0.0,67.53,0.0130,-0.0131,-0.0224
9,0.65,14.7,0.3694,4.29,1006.9,42.74,0.1201,0.0,67.10,0.0098,-0.0163,-0.0256


## 5. Controlled Feature Ablation — Stage 2 Only
Evaluates 10 controlled configurations under identical GroupKFold splits to determine whether Stage-2 signal is multi-family or driven by a single feature family.


In [6]:
df_feature_abl = run_stage2_controlled_ablations(df_merged, seed=42)
df_feature_abl.to_csv(OUTPUT_DIR / "feature_ablation.csv", index=False)
display(df_feature_abl)


,ablation_configuration,feature_count,roc_auc,pr_auc,balanced_acc,policy_mean_f1,policy_mean_k,mean_context_tokens,context_reduction_pct_vs_k8,mean_regret
0,1. All Stage-2 Features,5,0.6503,0.3199,0.6448,0.3842,4.82,1115.9,36.54,0.1053
1,2. BM25 Retrieval Only,1,0.6432,0.3154,0.6030,0.3750,4.78,1107.0,37.05,0.1145
2,3. Semantic Only,1,0.6159,0.2479,0.5984,0.3722,4.88,1128.5,35.83,0.1172
3,4. Lexical Coverage Only,1,0.6406,0.2911,0.6418,0.3810,4.79,1108.8,36.95,0.1085
4,5. BM25 + Semantic,2,0.6391,0.2819,0.6176,0.3801,4.87,1124.9,36.03,0.1095
5,6. BM25 + Lexical,2,0.6639,0.3664,0.6145,0.3800,4.83,1117.7,36.44,0.1095
6,7. Semantic + Lexical,2,0.6508,0.3096,0.6038,0.3747,4.87,1124.9,36.03,0.1148
7,8. Remove BM25,4,0.6404,0.2848,0.6129,0.3819,4.97,1146.3,34.81,0.1076
8,9. Remove Semantic,4,0.6504,0.3323,0.6118,0.3745,4.84,1119.5,36.34,0.1150
9,10. Remove Lexical,4,0.6305,0.2925,0.6091,0.3777,4.85,1121.3,36.23,0.1118


## 6. Feature Importance & Interpretability
Examines standardized coefficients across folds, odds ratios, and directions of association.


In [7]:
df_coefs = res_primary["df_coefficients"]
df_coefs.to_csv(OUTPUT_DIR / "feature_coefficients.csv", index=False)
display(df_coefs)


,feature,mean_coefficient,std_coefficient,odds_ratio,direction
2,lex_coverage_top2,0.3794,0.1274,1.4614,Positive (Expands)
1,bm25_mean_score,0.2818,0.0546,1.3255,Positive (Expands)
0,sem_sim_mean_top6,0.1768,0.1237,1.1934,Positive (Expands)
3,evidence_query_cluster_span,0.1467,0.1006,1.1580,Positive (Expands)
4,evidence_lexical_overlap_mean,-0.0087,0.1119,0.9914,Negative (Halts)


## 7. Confounder & Proxy Variable Analysis
Tests whether Stage-2 feature signals survive controlling for obvious correlated confounders (query length, passage length, BM25 vs Semantic similarity).


In [8]:
# Compute correlations with confounders
confounders = ["query_word_count", "bm25_mean_score", "sem_sim_mean_top6", "lex_coverage_top2"]
df_corr_conf = df_merged[confounders + ["G_4_6"]].corr(method="spearman")
print("Spearman Correlation Matrix with Marginal Gain G_4_6 and Potential Confounders:")
display(df_corr_conf)

# Check partial correlation of sem_sim_mean_top6 with G_4_6 partialling out query_word_count and bm25_mean_score
from scipy import stats

def partial_corr_spearman(df, x_col, y_col, covar_cols):
    from sklearn.linear_model import LinearRegression
    X_cov = df[covar_cols].values
    x_res = df[x_col].values - LinearRegression().fit(X_cov, df[x_col].values).predict(X_cov)
    y_res = df[y_col].values - LinearRegression().fit(X_cov, df[y_col].values).predict(X_cov)
    return stats.spearmanr(x_res, y_res)

rho_sem_adj, p_sem_adj = partial_corr_spearman(df_merged, "sem_sim_mean_top6", "G_4_6", ["query_word_count", "bm25_mean_score"])
rho_bm25_adj, p_bm25_adj = partial_corr_spearman(df_merged, "bm25_mean_score", "G_4_6", ["query_word_count"])
rho_lex_adj, p_lex_adj = partial_corr_spearman(df_merged, "lex_coverage_top2", "G_4_6", ["query_word_count", "bm25_mean_score"])

print(f"Partial Spearman rho for sem_sim_mean_top6 controlling for query length and BM25: {rho_sem_adj:+.4f} (p={p_sem_adj:.4f})")
print(f"Partial Spearman rho for bm25_mean_score controlling for query length: {rho_bm25_adj:+.4f} (p={p_bm25_adj:.4f})")
print(f"Partial Spearman rho for lex_coverage_top2 controlling for query length and BM25: {rho_lex_adj:+.4f} (p={p_lex_adj:.4f})")


Spearman Correlation Matrix with Marginal Gain G_4_6 and Potential Confounders:


,query_word_count,bm25_mean_score,sem_sim_mean_top6,lex_coverage_top2,G_4_6
query_word_count,1.000000,0.598750,0.519099,0.076727,0.103661
bm25_mean_score,0.598750,1.000000,0.558007,0.280604,0.174339
sem_sim_mean_top6,0.519099,0.558007,1.000000,0.301888,0.172086
lex_coverage_top2,0.076727,0.280604,0.301888,1.000000,0.167254
G_4_6,0.103661,0.174339,0.172086,0.167254,1.000000


Partial Spearman rho for sem_sim_mean_top6 controlling for query length and BM25: +0.0497 (p=0.4520)
Partial Spearman rho for bm25_mean_score controlling for query length: +0.0930 (p=0.1590)
Partial Spearman rho for lex_coverage_top2 controlling for query length and BM25: +0.0401 (p=0.5438)


## 8. Subgroup Analysis
Evaluates Stage-2 performance across query types: numerical, definitional, complexity levels, coverage levels, and evidence dispersion.


In [9]:
df_subgroups = evaluate_stage2_subgroups(df_merged, df_stage2_oof)
df_subgroups.to_csv(OUTPUT_DIR / "subgroup_analysis.csv", index=False)
display(df_subgroups)


,subgroup,query_count,actual_positive_rate_pct,policy_expansion_rate_pct,roc_auc,mean_f1,mean_regret
0,Numerical Queries,5,0.0,80.0,N/A,0.0000,0.0000
1,Non-Numerical Queries,226,19.9,40.3,0.6562,0.3927,0.1077
2,What/Which Queries,128,21.1,43.0,0.6337,0.3757,0.1235
3,Other Syntax Queries,103,17.5,38.8,0.6641,0.3947,0.0827
4,High Complexity (Conjunction > median),27,29.6,63.0,0.5724,0.4600,0.1235
5,Low Complexity (Conjunction <= median),204,18.1,38.2,0.6483,0.3741,0.1029
6,High Lexical Coverage (>= median),136,23.5,61.0,0.694,0.3999,0.0855
7,Low Lexical Coverage (< median),95,13.7,12.6,0.4765,0.3616,0.1338
8,High Semantic Tail Sim (>= median),116,25.9,63.8,0.5909,0.3674,0.0981
9,Low Semantic Tail Sim (< median),115,13.0,18.3,0.5567,0.4011,0.1127


## 9. Granular Stage-2 Error Analysis
Decomposes Stage-2 decisions into False Expansions ($k=6$ chosen when $k=4$ was better) vs False Stoppings ($k=4$ chosen when $k=6$ was better).


In [10]:
df_audit, df_error_summary = perform_stage2_error_analysis(df_merged, df_stage2_oof)
df_error_summary.to_csv(OUTPUT_DIR / "error_analysis.csv", index=False)
display(df_error_summary)


,error_category,query_count,percentage,mean_actual_gain,mean_bm25_score,mean_sem_sim_top6,mean_lex_coverage_top2,mean_query_length
2,True Negative (Correct Halt),120,51.95,-0.0477,0.99,0.601,0.512,7.5
0,False Expansion (Unneeded Context),66,28.57,-0.0632,1.66,0.667,0.811,9.5
3,True Positive (Beneficial Expansion),29,12.55,0.3395,1.77,0.676,0.892,9.7
1,False Stop (Missed Gain),16,6.93,0.3802,1.09,0.611,0.508,7.0


## 10. Statistical Comparison with Static Baselines (Paper-Clustered Paired Bootstrap)
Conducts $B=2,000$ paper-clustered bootstrap replicates to evaluate differences in Answer F1, Context Tokens, and Regret vs Static $k \in \{4, 6, 8\}$.


In [11]:
df_stat_comp = run_stage2_statistical_comparisons(
    df_merged,
    res_primary["df_policy_results"],
    n_bootstrap=2000,
    seed=42,
)
df_stat_comp.to_csv(OUTPUT_DIR / "statistical_comparisons.csv", index=False)
display(df_stat_comp)


,comparison,mean_delta_f1,ci_95_f1_low,ci_95_f1_high,statistically_detectable_f1_diff,mean_delta_tokens,ci_95_tokens_low,ci_95_tokens_high,mean_delta_regret,ci_95_regret_low,ci_95_regret_high
0,Policy C1 vs Static k=4,0.0246,-0.0017,0.0516,False,169.8,136.3,203.0,-0.0245,-0.0516,0.0017
1,Policy C1 vs Static k=6,-0.0015,-0.0241,0.0195,False,-243.2,-276.6,-211.0,0.0016,-0.0194,0.0241
2,Policy C1 vs Static k=8,-0.0108,-0.0451,0.0225,False,-642.6,-676.6,-610.8,0.0109,-0.0225,0.0451


## 11. Multi-Objective Pareto Frontier Analysis
Maps Policy C1 against static baselines ($k=2, 4, 5, 6, 8$), Random Allocation, QCCA-V1, and the Oracle Sequential ceiling on the context-vs-quality surface.


In [12]:
all_systems = [
    {"system": "Static k=2", "mean_f1": float(df_merged["F1_k2"].mean()), "mean_context_tokens": CONTEXT_TOKEN_COSTS[2], "context_reduction_pct_vs_k8": 69.60},
    {"system": "Static k=4", "mean_f1": float(df_merged["F1_k4"].mean()), "mean_context_tokens": CONTEXT_TOKEN_COSTS[4], "context_reduction_pct_vs_k8": 46.20},
    {"system": "Static k=5", "mean_f1": float(df_merged["F1_k5"].mean()) if "F1_k5" in df_merged else 0.3629, "mean_context_tokens": CONTEXT_TOKEN_COSTS[5], "context_reduction_pct_vs_k8": 34.21},
    {"system": "Static k=6", "mean_f1": float(df_merged["F1_k6"].mean()), "mean_context_tokens": CONTEXT_TOKEN_COSTS[6], "context_reduction_pct_vs_k8": 22.71},
    {"system": "Static k=8", "mean_f1": float(df_merged["F1_k8"].mean()), "mean_context_tokens": CONTEXT_TOKEN_COSTS[8], "context_reduction_pct_vs_k8": 0.00},
    {"system": "Random Allocation", "mean_f1": 0.3709, "mean_context_tokens": 1131.0, "context_reduction_pct_vs_k8": 35.68},
    {"system": "QCCA-V1 (Decision Tree)", "mean_f1": 0.3282, "mean_context_tokens": 671.0, "context_reduction_pct_vs_k8": 61.84},
    {"system": "QCCA-V1 (Rule)", "mean_f1": 0.3286, "mean_context_tokens": 854.4, "context_reduction_pct_vs_k8": 51.41},
    {"system": "QCCA-V2 Policy C1 (Final)", "mean_f1": res_primary["policy_summary"]["mean_f1"], "mean_context_tokens": res_primary["policy_summary"]["mean_context_tokens"], "context_reduction_pct_vs_k8": res_primary["policy_summary"]["context_reduction_pct_vs_k8"]},
    {"system": "Oracle Sequential Ceiling", "mean_f1": 0.4069, "mean_context_tokens": 650.3, "context_reduction_pct_vs_k8": 63.02},
    {"system": "Epsilon Oracle (epsilon=0.01)", "mean_f1": float(df_merged["oracle_f1"].mean()), "mean_context_tokens": 841.3, "context_reduction_pct_vs_k8": 52.16},
]

df_pareto = compute_pareto_frontier(pd.DataFrame(all_systems))
df_pareto.to_csv(OUTPUT_DIR / "policy_pareto_points.csv", index=False)
print("Pareto Status of Candidate Systems:")
display(df_pareto[["system", "mean_f1", "mean_context_tokens", "context_reduction_pct_vs_k8", "is_pareto_optimal"]])


Pareto Status of Candidate Systems:


,system,mean_f1,mean_context_tokens,context_reduction_pct_vs_k8,is_pareto_optimal
0,Static k=2,0.303835,534.5,69.60,True
1,Oracle Sequential Ceiling,0.406900,650.3,63.02,True
2,QCCA-V1 (Decision Tree),0.328200,671.0,61.84,False
3,Epsilon Oracle (epsilon=0.01),0.489391,841.3,52.16,True
4,QCCA-V1 (Rule),0.328600,854.4,51.41,False
5,Static k=4,0.359601,946.1,46.20,False
6,QCCA-V2 Policy C1 (Final),0.384200,1115.9,36.54,False
7,Random Allocation,0.370900,1131.0,35.68,False
8,Static k=5,0.362900,1157.0,34.21,False
9,Static k=6,0.385699,1359.1,22.71,False


## 12. Publication-Grade Figure Generation
Generates all 9 validation figures into `results/week4/qcca_v2_final_validation/figures/`.


In [13]:
print("Generating all 9 publication-grade validation figures...")
generate_stage2_validation_figures(
    df_merged=df_merged,
    res_eval=res_primary,
    df_thresh=df_threshold_sens,
    df_abl=df_feature_abl,
    df_sg=df_subgroups,
    df_err_summary=df_error_summary,
    df_pareto=df_pareto,
    fig_dir=FIG_DIR,
)
print(f"All figures generated successfully in {FIG_DIR}")


Generating all 9 publication-grade validation figures...
All figures generated successfully in /home/gunavenkat/Downloads/CS787-RAG-Project/results/week4/qcca_v2_final_validation/figures


## 13. Final Validation Summary


In [14]:
print("=" * 80)
print("QCCA-V2 FINAL STAGE-2 VALIDATION COMPLETE")
print("=" * 80)
print(f"Artifact directory: {OUTPUT_DIR}")
print("Generated CSVs:")
for p in sorted(OUTPUT_DIR.glob("*.csv")):
    print(f"  - {p.name}")
print("\nGenerated Figures:")
for p in sorted(FIG_DIR.glob("*.png")):
    print(f"  - {p.name}")
print("=" * 80)


QCCA-V2 FINAL STAGE-2 VALIDATION COMPLETE
Artifact directory: /home/gunavenkat/Downloads/CS787-RAG-Project/results/week4/qcca_v2_final_validation
Generated CSVs:
  - calibration_results.csv
  - error_analysis.csv
  - feature_ablation.csv
  - feature_coefficients.csv
  - policy_pareto_points.csv
  - stage2_oof_predictions.csv
  - statistical_comparisons.csv
  - subgroup_analysis.csv
  - threshold_sensitivity.csv

Generated Figures:
  - calibration_curve.png
  - error_analysis.png
  - feature_ablation.png
  - feature_coefficients.png
  - feature_response_curves.png
  - policy_pareto_frontier.png
  - stage2_roc_pr.png
  - subgroup_performance.png
  - threshold_quality_cost.png
